# EDA for Physiological Signals and Annotations

This notebook explores the Kaggle PhysioNet dataset, demographics, EDF signals, CAISR annotations, and human annotations.

In [ ]:
import os
import glob
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from sklearn.metrics import cohen_kappa_score
import warnings

warnings.filterwarnings('ignore')

# Setup paths
BASE_DIR = "data/kaggle_raw/training_set"
FIGURES_DIR = "notebooks/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# 1. Load demographics data
demo_path = os.path.join(BASE_DIR, "demographics.csv")
df = pl.read_csv(demo_path)

print("Shape:", df.shape)
print("Columns:", df.columns)

# Cognitive_Impairment distribution (counts and %)
ci_dist = df['Cognitive_Impairment'].value_counts().sort_by('Cognitive_Impairment')
ci_dist_pct = ci_dist.with_columns(pl.col('count').alias('percentage') * 100)
print("\nCognitive_Impairment Distribution:")
print(ci_dist_pct)

# Time_to_Event distribution for True cases
true_mask = df['Cognitive_Impairment'] == True
time_true = df.filter(true_mask)['Time_to_Event'].to_numpy()

plt.figure(figsize=(8, 5))
sns.histplot(time_true, bins=30, kde=True)
plt.title('Distribution of Time_to_Event for Cognitive_Impairment=True')
plt.xlabel('Time to Event')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'time_to_event_true.png'))
plt.show()

# Site distribution
print("\nSite Distribution:")
print(df['Site'].value_counts())

# Age distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'], bins=30, kde=True)
plt.title('Distribution of Age')
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'age_distribution.png'))
plt.show()

# Sex distribution
print("\nSex Distribution:")
print(df['Sex'].value_counts())

In [ ]:
# 2. Pick the first EDF file found in S0001 and load with MNE
edf_dir = os.path.join(BASE_DIR, "physiological_data", "S0001")
edf_files = glob.glob(os.path.join(edf_dir, "*.edf"))
if not edf_files:
    raise FileNotFoundError(f"No EDF files found in {edf_dir}")

first_edf = edf_files[0]
print(f"Loading EDF file: {os.path.basename(first_edf)}")

raw = mne.io.read_raw_edf(first_edf, preload=True)
print("\nChannel names:", raw.ch_names)
print("Channel types:", [mne.io.pick.channel_type(raw.get_channel_types(), i) for i in range(len(raw.ch_names))])
print("Sampling rate (Hz):", raw.info['sfreq'])
print("Recording duration (s):", raw.times[-1])

# Save a 30-second epoch plot of all channels
duration = 30
start_time = 0
end_time = start_time + duration
data, times = raw[start_time:end_time]
ch_names = raw.ch_names

plt.figure(figsize=(12, 8))
for i, ch in enumerate(ch_names):
    plt.subplot(len(ch_names), 1, i+1)
    plt.plot(times, data[i])
    plt.title(f'{ch}')
    plt.ylabel('Amplitude')
    plt.xlabel('Time (s)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'edf_30s_epoch.png'))
plt.show()

In [ ]:
# 3. Load the corresponding CAISR annotation EDF
caiser_dir = "algorithmic_annotations/S0001"
caiser_files = glob.glob(os.path.join(caiser_dir, "*.edf"))
if not caiser_files:
    raise FileNotFoundError(f"No CAISR annotation files found in {caiser_dir}")

caiser_edf = caiser_files[0]
print(f"Loading CAISR EDF: {os.path.basename(caiser_edf)}")

raw_caiser = mne.io.read_raw_edf(caiser_edf, preload=True)
ch_names_caiser = raw_caiser.ch_names
print("CAISR Channels:", ch_names_caiser)

# Identify sleep stage and arousal channels (fallback to first two if naming is ambiguous)
stage_ch = [ch for ch in ch_names_caiser if 'Stage' in ch.upper() or 'Sleep' in ch.upper()]
arousal_ch = [ch for ch in ch_names_caiser if 'Arousal' in ch.upper()]
if not stage_ch:
    stage_ch = [ch_names_caiser[0]]
if not arousal_ch and len(ch_names_caiser) > 1:
    arousal_ch = [ch_names_caiser[1]]

duration_caiser = raw_caiser.times[-1]
t_plot = np.linspace(0, duration_caiser, len(raw_caiser[stage_ch[0]][0][0]))
stage_data, _ = raw_caiser[stage_ch[0]]
arousal_data, _ = raw_caiser[arousal_ch[0]] if arousal_ch else (np.zeros_like(stage_data[0]), None)

# Full-night sleep stage hypnogram plot
plt.figure(figsize=(14, 6))
plt.subplot(2, 1, 1)
plt.step(t_plot, stage_data[0], where='post', color='blue')
plt.title('Full-Night Sleep Stage Hypnogram (CAISR)')
plt.ylabel('Sleep Stage')
plt.ylim(-0.5, max(stage_data[0].max(), 4.5))
plt.grid(True)

# Arousal events overlaid on hypnogram
if arousal_ch:
    plt.subplot(2, 1, 2)
    plt.step(t_plot, arousal_data[0], where='post', color='red')
    plt.title('Arousal Events Overlaid on Hypnogram')
    plt.ylabel('Arousal')
    plt.xlabel('Time (s)')
    plt.ylim(-0.5, 1.5)
    plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'caiser_hypnogram_arousals.png'))
plt.show()

In [ ]:
# 4. Load the corresponding human annotation file and compute sleep staging agreement
human_dir = "human_annotations/S0001"
human_files = glob.glob(os.path.join(human_dir, "*"))
if not human_files:
    raise FileNotFoundError(f"No human annotation files found in {human_dir}. Adjust path if needed.")

# Assuming CSV format with a 'SleepStage' column matching CAISR's stage encoding
human_df = pl.read_csv(human_files[0])
print("Human Annotation Columns:", human_df.columns)

# Align lengths and extract labels for comparison
caiser_stage = stage_data[0]
human_stage = human_df['SleepStage'].to_numpy()  # Adjust column name if your dataset uses a different header

min_len = min(len(caiser_stage), len(human_stage))
caiser_labels = caiser_stage[:min_len].astype(int)
human_labels = human_stage[:min_len].astype(int)

kappa = cohen_kappa_score(caiser_labels, human_labels)
print(f"\nCohen's Kappa (CAISR vs Human): {kappa:.4f}")